In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install torch torchvision torchaudio --quiet
!pip install opencv-python tqdm tensorboard --quiet

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam, AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau

import cv2
import numpy as np
import os
import glob
from tqdm import tqdm
import random
import matplotlib.pyplot as plt

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")

In [ ]:
class ConvBNReLU(nn.Sequential):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1):
        super().__init__(
            nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = ConvBNReLU(channels, channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, 1, 1)
        self.bn = nn.BatchNorm2d(channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        residual = x
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.bn(x)
        x = x + residual
        x = self.relu(x)
        return x

class Encoder(nn.Module):
    def __init__(self, in_channels=4):
        super().__init__()
        self.conv1 = ConvBNReLU(in_channels, 32, kernel_size=3, stride=1, padding=1)
        self.down1 = ConvBNReLU(32, 64, kernel_size=3, stride=2, padding=1)
        self.down2 = ConvBNReLU(64, 128, kernel_size=3, stride=2, padding=1)
        self.down3 = ConvBNReLU(128, 256, kernel_size=3, stride=2, padding=1)
        self.down4 = ConvBNReLU(256, 512, kernel_size=3, stride=2, padding=1)

        self.res1 = ResidualBlock(64)
        self.res2 = ResidualBlock(128)
        self.res3 = ResidualBlock(256)
        self.res4 = ResidualBlock(512)

    def forward(self, x):
        features = {}

        x = self.conv1(x)

        x = self.down1(x)
        x = self.res1(x)
        features['level1'] = x

        x = self.down2(x)
        x = self.res2(x)
        features['level2'] = x

        x = self.down3(x)
        x = self.res3(x)
        features['level3'] = x

        x = self.down4(x)
        x = self.res4(x)
        features['level4'] = x

        return features

class Decoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.up4 = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            ConvBNReLU(512, 256)
        )
        self.conv4 = ResidualBlock(256)

        self.up3 = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            ConvBNReLU(256, 128)
        )
        self.conv3 = ResidualBlock(128)

        self.up2 = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            ConvBNReLU(128, 64)
        )
        self.conv2 = ResidualBlock(64)

        self.up1 = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            ConvBNReLU(64, 32)
        )
        self.conv1 = ResidualBlock(32)

        self.alpha_out = nn.Sequential(
            ConvBNReLU(32, 16),
            nn.Conv2d(16, 1, 3, 1, 1),
            nn.Sigmoid()
        )

    def forward(self, encoder_features):
        x = encoder_features['level4']

        x = self.up4(x)
        x = x + encoder_features['level3']
        x = self.conv4(x)

        x = self.up3(x)
        x = x + encoder_features['level2']
        x = self.conv3(x)

        x = self.up2(x)
        x = x + encoder_features['level1']
        x = self.conv2(x)

        x = self.up1(x)
        x = self.conv1(x)

        alpha = self.alpha_out(x)

        return alpha

class SemanticRefineNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = Encoder(in_channels=4)
        self.decoder = Decoder()

    def forward(self, rgb, base_alpha):
        x = torch.cat([rgb, base_alpha], dim=1)

        features = self.encoder(x)

        semantic_alpha = self.decoder(features)

        return semantic_alpha

In [ ]:
class RefineDataset(Dataset):
    def __init__(self, data_root, split='train', size=(512, 512),
                 use_confidence_weight=True):
        self.size = size
        self.use_confidence_weight = use_confidence_weight

        if split == 'train':
            self.origin_dir = os.path.join(data_root, 'train', 'blurred_image')
            self.mask_dir = os.path.join(data_root, 'train', 'mask')
            self.base_alpha_dir = os.path.join(data_root, 'train', 'base_alpha')
            self.sem_label_dir = os.path.join(data_root, 'train', 'sem_label')
        else:
            self.origin_dir = os.path.join(data_root, 'validation', 'P3M-500-P', 'blurred_image')
            self.mask_dir = os.path.join(data_root, 'validation', 'P3M-500-P', 'mask')
            self.base_alpha_dir = os.path.join(data_root, 'validation', 'P3M-500-P', 'base_alpha')
            self.sem_label_dir = os.path.join(data_root, 'validation', 'P3M-500-P', 'sem_label')

        self.image_paths = sorted(glob.glob(os.path.join(self.origin_dir, '*.*')))

        valid_paths = []
        for img_path in self.image_paths:
            name = os.path.basename(img_path).split('.')[0]

            mask_path = os.path.join(self.mask_dir, f"{name}.png")
            base_path = os.path.join(self.base_alpha_dir, f"{name}.png")

            if os.path.exists(mask_path) and os.path.exists(base_path):
                valid_paths.append(img_path)

        self.image_paths = valid_paths
        print(f" {split} dataset: {len(self.image_paths)} images")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        name = os.path.basename(img_path).split('.')[0]

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, self.size)
        image = image.astype(np.float32) / 255.0
        image = torch.from_numpy(image).permute(2, 0, 1)

        mask_path = os.path.join(self.mask_dir, f"{name}.png")
        gt_alpha = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        gt_alpha = cv2.resize(gt_alpha, self.size)
        gt_alpha = gt_alpha.astype(np.float32) / 255.0
        gt_alpha = torch.from_numpy(gt_alpha).unsqueeze(0)

        base_path = os.path.join(self.base_alpha_dir, f"{name}.png")
        base_alpha = cv2.imread(base_path, cv2.IMREAD_GRAYSCALE)
        base_alpha = cv2.resize(base_alpha, self.size)
        base_alpha = base_alpha.astype(np.float32) / 255.0
        base_alpha = torch.from_numpy(base_alpha).unsqueeze(0)

        sem_label = None
        if self.use_confidence_weight:
            sem_path = os.path.join(self.sem_label_dir, f"{name}.png")
            if os.path.exists(sem_path):
                sem_label = cv2.imread(sem_path, cv2.IMREAD_GRAYSCALE)
                sem_label = cv2.resize(sem_label, self.size, interpolation=cv2.INTER_NEAREST)
                sem_label = torch.from_numpy(sem_label).long()
            else:
                alpha_np = gt_alpha.squeeze(0).numpy()
                sem_label = np.zeros_like(alpha_np, dtype=np.int64)
                sem_label[(alpha_np > 0.05) & (alpha_np <= 0.5)] = 1
                sem_label[(alpha_np > 0.5) & (alpha_np < 0.95)] = 2
                sem_label[alpha_np >= 0.95] = 3
                sem_label = torch.from_numpy(sem_label).long()

        result = {
            'image': image,
            'gt_alpha': gt_alpha,
            'base_alpha': base_alpha,
            'name': name
        }

        if sem_label is not None:
            result['sem_label'] = sem_label

        return result

In [ ]:
class TransitionFocusedLoss(nn.Module):
    def __init__(self, transition_weight=3.0, certain_weight=0.2):
        super().__init__()
        self.transition_weight = transition_weight
        self.certain_weight = certain_weight

    def forward(self, pred_alpha, gt_alpha, base_alpha=None, sem_label=None):
        l1_loss = F.l1_loss(pred_alpha, gt_alpha, reduction='none')

        if sem_label is not None:
            weight = torch.ones_like(pred_alpha)

            transition_mask = (sem_label == 1) | (sem_label == 2)
            weight[transition_mask.unsqueeze(1)] = self.transition_weight

            certain_mask = (sem_label == 0) | (sem_label == 3)
            weight[certain_mask.unsqueeze(1)] = self.certain_weight

        else:
            transition_mask = (gt_alpha > 0.05) & (gt_alpha < 0.95)
            weight = torch.where(transition_mask,
                                 torch.tensor(self.transition_weight).to(pred_alpha.device),
                                 torch.tensor(self.certain_weight).to(pred_alpha.device))

        weighted_loss = (l1_loss * weight).mean()

        if transition_mask.any() if sem_label is None else transition_mask.any():
            dx = torch.abs(pred_alpha[:, :, :, 1:] - pred_alpha[:, :, :, :-1])
            dy = torch.abs(pred_alpha[:, :, 1:, :] - pred_alpha[:, :, :-1, :])
            smooth_loss = (dx.mean() + dy.mean()) * 0.1
            weighted_loss = weighted_loss + smooth_loss

        return weighted_loss


class HybridLoss(nn.Module):
    def __init__(self, transition_weight=3.0, certain_weight=0.2, grad_weight=0.1):
        super().__init__()
        self.transition_weight = transition_weight
        self.certain_weight = certain_weight
        self.grad_weight = grad_weight

    def forward(self, pred_alpha, gt_alpha, base_alpha=None, sem_label=None):
        l1_loss = F.l1_loss(pred_alpha, gt_alpha, reduction='none')

        if sem_label is not None:
            weight = torch.ones_like(pred_alpha)
            transition_mask = (sem_label == 1) | (sem_label == 2)
            weight[transition_mask.unsqueeze(1)] = self.transition_weight
            certain_mask = (sem_label == 0) | (sem_label == 3)
            weight[certain_mask.unsqueeze(1)] = self.certain_weight
        else:
            transition_mask = (gt_alpha > 0.05) & (gt_alpha < 0.95)
            weight = torch.where(transition_mask,
                                 torch.tensor(self.transition_weight).to(pred_alpha.device),
                                 torch.tensor(self.certain_weight).to(pred_alpha.device))

        weighted_l1 = (l1_loss * weight).mean()

        if (sem_label is not None and transition_mask.any()) or (sem_label is None and transition_mask.any()):
            pred_grad_x = torch.abs(pred_alpha[:, :, :, 1:] - pred_alpha[:, :, :, :-1])
            pred_grad_y = torch.abs(pred_alpha[:, :, 1:, :] - pred_alpha[:, :, :-1, :])
            gt_grad_x = torch.abs(gt_alpha[:, :, :, 1:] - gt_alpha[:, :, :, :-1])
            gt_grad_y = torch.abs(gt_alpha[:, :, 1:, :] - gt_alpha[:, :, :-1, :])

            grad_loss = (F.l1_loss(pred_grad_x, gt_grad_x, reduction='mean') +
                        F.l1_loss(pred_grad_y, gt_grad_y, reduction='mean'))
        else:
            grad_loss = torch.tensor(0.0).to(pred_alpha.device)

        total_loss = weighted_l1 + self.grad_weight * grad_loss

        return total_loss

In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, device, epoch):
    model.train()
    total_loss = 0

    pbar = tqdm(dataloader, desc=f"Epoch {epoch}")
    for batch in pbar:
        images = batch['image'].to(device)
        base_alpha = batch['base_alpha'].to(device)
        gt_alpha = batch['gt_alpha'].to(device)

        sem_label = batch.get('sem_label', None)
        if sem_label is not None:
            sem_label = sem_label.to(device)

        pred_alpha = model(images, base_alpha)

        loss = criterion(pred_alpha, gt_alpha, base_alpha, sem_label)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        pbar.set_postfix({'loss': f"{loss.item():.4f}"})

    return total_loss / len(dataloader)


def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0

    total_transition_error = 0
    total_certain_error = 0
    total_pixels = 0
    total_transition_pixels = 0

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Validating"):
            images = batch['image'].to(device)
            base_alpha = batch['base_alpha'].to(device)
            gt_alpha = batch['gt_alpha'].to(device)
            sem_label = batch.get('sem_label', None)
            if sem_label is not None:
                sem_label = sem_label.to(device)

            pred_alpha = model(images, base_alpha)
            loss = criterion(pred_alpha, gt_alpha, base_alpha, sem_label)
            total_loss += loss.item()

            pred_np = pred_alpha.cpu().numpy()
            gt_np = gt_alpha.cpu().numpy()

            transition_mask = (gt_np > 0.05) & (gt_np < 0.95)
            transition_error = np.abs(pred_np - gt_np)[transition_mask].mean() if transition_mask.any() else 0
            total_transition_error += transition_error * transition_mask.sum()
            total_transition_pixels += transition_mask.sum()

            certain_mask = (gt_np <= 0.05) | (gt_np >= 0.95)
            certain_error = np.abs(pred_np - gt_np)[certain_mask].mean() if certain_mask.any() else 0
            total_certain_error += certain_error * certain_mask.sum()
            total_pixels += certain_mask.sum()

    avg_transition_error = total_transition_error / total_transition_pixels if total_transition_pixels > 0 else 0
    avg_certain_error = total_certain_error / total_pixels if total_pixels > 0 else 0

    return {
        'loss': total_loss / len(dataloader),
        'transition_error': avg_transition_error,
        'certain_error': avg_certain_error
    }

In [ ]:
def visualize_results(model, dataloader, device, num_samples=4, save_path=None):
    model.eval()

    samples = []
    for i, batch in enumerate(dataloader):
        if len(samples) >= num_samples:
            break
        sample = {
            'image': batch['image'][0:1],
            'base_alpha': batch['base_alpha'][0:1],
            'gt_alpha': batch['gt_alpha'][0:1],
            'name': batch['name'][0] if 'name' in batch else str(i)
        }
        samples.append(sample)

    fig, axes = plt.subplots(num_samples, 4, figsize=(16, 4 * num_samples))
    if num_samples == 1:
        axes = axes.reshape(1, -1)

    with torch.no_grad():
        for i, sample in enumerate(samples):
            images = sample['image'].to(device)
            base_alpha = sample['base_alpha'].to(device)
            gt_alpha = sample['gt_alpha'].cpu().numpy().squeeze()

            pred_alpha = model(images, base_alpha)
            pred_alpha = pred_alpha.cpu().numpy().squeeze()
            base_alpha_np = base_alpha.cpu().numpy().squeeze()

            img_np = images[0].cpu().permute(1, 2, 0).numpy()

            axes[i, 0].imshow(img_np)
            axes[i, 0].set_title(f'Original', fontsize=10)
            axes[i, 0].axis('off')

            axes[i, 1].imshow(base_alpha_np, cmap='gray', vmin=0, vmax=1)
            axes[i, 1].set_title('Base Alpha (RVM)', fontsize=10)
            axes[i, 1].axis('off')

            axes[i, 2].imshow(pred_alpha, cmap='gray', vmin=0, vmax=1)
            axes[i, 2].set_title('Predicted Alpha', fontsize=10)
            axes[i, 2].axis('off')

            axes[i, 3].imshow(gt_alpha, cmap='gray', vmin=0, vmax=1)
            axes[i, 3].set_title('Ground Truth', fontsize=10)
            axes[i, 3].axis('off')

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved visualization results to: {save_path}")


def compute_refinement_metrics(model, dataloader, device):
    model.eval()

    total_rvm_transition_error = 0
    total_our_transition_error = 0
    total_transition_pixels = 0

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Computing metrics"):
            images = batch['image'].to(device)
            base_alpha = batch['base_alpha'].to(device)
            gt_alpha = batch['gt_alpha'].cpu().numpy().squeeze()

            pred_alpha = model(images, base_alpha)
            pred_alpha = pred_alpha.cpu().numpy().squeeze()
            base_alpha_np = base_alpha.cpu().numpy().squeeze()

            transition_mask = (gt_alpha > 0.05) & (gt_alpha < 0.95)

            if transition_mask.any():
                rvm_error = np.abs(base_alpha_np - gt_alpha)[transition_mask].sum()
                our_error = np.abs(pred_alpha - gt_alpha)[transition_mask].sum()

                total_rvm_transition_error += rvm_error
                total_our_transition_error += our_error
                total_transition_pixels += transition_mask.sum()

    avg_rvm_error = total_rvm_transition_error / total_transition_pixels
    avg_our_error = total_our_transition_error / total_transition_pixels
    improvement = (avg_rvm_error - avg_our_error) / avg_rvm_error * 100

    print(f"\nTransition Region Refinement Metrics:")
    print(f"   RVM average error: {avg_rvm_error:.4f}")
    print(f"   Our model error: {avg_our_error:.4f}")
    print(f"   Improvement: {improvement:.2f}%")

    return {
        'rvm_error': avg_rvm_error,
        'our_error': avg_our_error,
        'improvement': improvement
    }

In [ ]:
def main():
    DATA_ROOT = "/content/drive/MyDrive/DATA/P3M-10k"
    SAVE_DIR = "/content/drive/MyDrive/RVM/semantic_refine_model"

    BATCH_SIZE = 8
    NUM_EPOCHS = 20
    LEARNING_RATE = 1e-4
    NUM_WORKERS = 4
    IMG_SIZE = (512, 512)

    TEST_MODE = True
    TEST_SAMPLES = 9421

    TRANSITION_WEIGHT = 3.0
    CERTAIN_WEIGHT = 0.2

    os.makedirs(SAVE_DIR, exist_ok=True)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    print("\nLoading training dataset...")
    train_dataset = RefineDataset(DATA_ROOT, split='train', size=IMG_SIZE, use_confidence_weight=True)

    if TEST_MODE:
        print(f"Test mode: using only first {TEST_SAMPLES} samples")
        from torch.utils.data import Subset
        train_dataset = Subset(train_dataset, range(min(TEST_SAMPLES, len(train_dataset))))

    print("Loading validation dataset...")
    val_dataset = RefineDataset(DATA_ROOT, split='val', size=IMG_SIZE, use_confidence_weight=True)

    if TEST_MODE and len(val_dataset) > TEST_SAMPLES:
        val_dataset = Subset(val_dataset, range(TEST_SAMPLES))

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, drop_last=True)

    print(f"\nTraining batches: {len(train_loader)}, Validation batches: {len(val_loader)}")

    print("\nBuilding Semantic Refine Network...")
    model = SemanticRefineNet().to(device)

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"   Total parameters: {total_params:,}")
    print(f"   Trainable parameters: {trainable_params:,}")

    criterion = HybridLoss(
        transition_weight=TRANSITION_WEIGHT,
        certain_weight=CERTAIN_WEIGHT,
        grad_weight=0.1
    )

    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, min_lr=1e-6)

    print("\nStarting training!")
    print("=" * 60)

    best_val_loss = float('inf')
    history = {'train_loss': [], 'val_loss': [], 'val_transition_error': [], 'val_certain_error': []}

    for epoch in range(1, NUM_EPOCHS + 1):
        print(f"\nEpoch {epoch}/{NUM_EPOCHS}")
        print("-" * 40)

        train_loss = train_epoch(model, train_loader, criterion, optimizer, device, epoch)

        val_stats = validate(model, val_loader, criterion, device)

        scheduler.step(val_stats['loss'])

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_stats['loss'])
        history['val_transition_error'].append(val_stats['transition_error'])
        history['val_certain_error'].append(val_stats['certain_error'])

        print(f"\nEpoch {epoch} Results:")
        print(f"   Training loss: {train_loss:.4f}")
        print(f"   Validation loss: {val_stats['loss']:.4f}")
        print(f"   Transition error: {val_stats['transition_error']:.4f}")
        print(f"   Certain error: {val_stats['certain_error']:.4f}")
        print(f"   Learning rate: {scheduler.get_last_lr()[0]:.6f}")

        if epoch % 4 == 0:
            checkpoint_path = os.path.join(SAVE_DIR, f"model_epoch_{epoch}.pth")
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': val_stats['loss'],
                'train_loss': train_loss
            }, checkpoint_path)
            print(f"Model saved: {checkpoint_path}")

        if val_stats['loss'] < best_val_loss:
            best_val_loss = val_stats['loss']
            best_path = os.path.join(SAVE_DIR, "model_best.pth")
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'val_loss': val_stats['loss']
            }, best_path)
            print(f"Best model updated!")

        if epoch % 5 == 0:
            visualize_results(model, val_loader, device, num_samples=2,
                            save_path=os.path.join(SAVE_DIR, f"vis_epoch_{epoch}.png"))

    print("\n" + "=" * 60)
    print("Training complete!")
    print(f"All models saved in: {SAVE_DIR}")

    print("\nFinal evaluation on validation set:")
    metrics = compute_refinement_metrics(model, val_loader, device)

    import json
    with open(os.path.join(SAVE_DIR, "training_history.json"), 'w') as f:
        json.dump(history, f, indent=2)

    return model, history, metrics


if __name__ == "__main__":
    model, history, metrics = main()

In [ ]:
DATA_ROOT = "/content/drive/MyDrive/DATA/P3M-10k"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
import matplotlib.pyplot as plt
import random

def show_random_predictions(model_path, val_loader, device, num_samples=4):
    model = SemanticRefineNet().to(device)
    checkpoint = torch.load(model_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    print(f"Loaded model: {model_path}")

    indices = random.sample(range(len(val_loader.dataset)), min(num_samples, len(val_loader.dataset)))

    fig, axes = plt.subplots(num_samples, 4, figsize=(16, 4 * num_samples))
    if num_samples == 1:
        axes = axes.reshape(1, -1)

    with torch.no_grad():
        for row, idx in enumerate(indices):
            sample = val_loader.dataset[idx]

            image = sample['image'].unsqueeze(0).to(device)
            base_alpha = sample['base_alpha'].unsqueeze(0).to(device)
            gt_alpha = sample['gt_alpha'].squeeze().numpy()

            pred_alpha = model(image, base_alpha)
            pred_alpha = pred_alpha.squeeze().cpu().numpy()
            base_alpha_np = base_alpha.squeeze().cpu().numpy()

            img_np = sample['image'].permute(1, 2, 0).numpy()
            axes[row, 0].imshow(img_np)
            axes[row, 0].set_title('Original', fontsize=10)
            axes[row, 0].axis('off')

            axes[row, 1].imshow(base_alpha_np, cmap='gray', vmin=0, vmax=1)
            axes[row, 1].set_title('RVM (Base)', fontsize=10)
            axes[row, 1].axis('off')

            axes[row, 2].imshow(pred_alpha, cmap='gray', vmin=0, vmax=1)
            axes[row, 2].set_title('Ours (Refined)', fontsize=10)
            axes[row, 2].axis('off')

            axes[row, 3].imshow(gt_alpha, cmap='gray', vmin=0, vmax=1)
            axes[row, 3].set_title('Ground Truth', fontsize=10)
            axes[row, 3].axis('off')

    plt.tight_layout()
    plt.show()

val_dataset = RefineDataset(DATA_ROOT, split='val', size=(512, 512), use_confidence_weight=True)
val_loader_simple = DataLoader(val_dataset, batch_size=1, shuffle=False)

show_random_predictions(
    model_path="/content/drive/MyDrive/RVM/semantic_refine_model/model_best.pth",
    val_loader=val_loader_simple,
    device=device,
    num_samples=4
)